In [1]:
""""""""""""""
#Our model contains the following key layers: cropping ->segmentation -> EfficientNetv2 -> GRU -> output
#For efficient preprocessing, we use seperate code for cropping and segmentation
#This section of the code is used to train and perform final inference with various CNN–RNN combinations (best performing: EfficientNetV2 + GRU with 128 units).
#It enables systematic large-scale ablation experiments across multiple backbone and temporal architecture configurations.
#⚠️⚠️⚠️⚠️⚠️⚠️⚠️
#The input is expected to be videos resized to 224x224, with striker+bat cropping applied, followed by striker+bat segmentation where the striker is colored blue and the bat is colored green (semi-transparent).
#⚠️⚠️⚠️⚠️⚠️⚠️⚠️

'!!!!!!!!!!!!!!!!!!!!!!'

In [ ]:

"""
Cell 1: Library Imports and Environment Setup
--------------------------------------------

This cell imports all required libraries and modules used throughout the project.
It sets up the environment for:

1. System utilities and memory management
2. Data manipulation and visualization
3. Video processing
4. Deep learning model construction (TensorFlow / Keras)
5. Pretrained CNN backbones (transfer learning)
6. Training utilities and callbacks
7. Model evaluation metrics
8. External model/file downloading via Hugging Face Hub

The imports are grouped by functionality for clarity and reproducibility.
"""

# =========================
# System & Utility Libraries
# =========================
import os                      # File system operations
import warnings                # Warning control
import gc                      # Garbage collection (useful for freeing GPU/CPU memory)

# =========================
# Core Scientific Libraries
# =========================
import numpy as np             # Numerical computations
import pandas as pd            # Data manipulation
import matplotlib.pyplot as plt
import seaborn as sns          # Visualization

# =========================
# Video Processing
# =========================
from decord import VideoReader  # Efficient video frame loading
from mpl_toolkits.axes_grid1 import ImageGrid  # Grid visualization for images

# =========================
# TensorFlow / Keras Setup
# =========================
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input, TimeDistributed, Flatten, Bidirectional, LSTM, GRU,
    Conv2D, Dense, Dropout, BatchNormalization,
    Concatenate, Average, Attention, Lambda
)

# =========================
# Optimizers & Callbacks
# =========================
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    Callback, CSVLogger, EarlyStopping,
    ReduceLROnPlateau, ModelCheckpoint
)

# =========================
# Pretrained CNN Backbones (Transfer Learning)
# =========================
# These architectures can be used as feature extractors or fine-tuned models.
from tensorflow.keras.applications import (
    NASNetMobile, ConvNeXtTiny,
    EfficientNetV2S, EfficientNetV2B3, EfficientNetB7,
    InceptionResNetV2, ResNet50, ResNet50V2, InceptionV3,
    VGG19, Xception,
    DenseNet121, DenseNet201,
    MobileNetV2, MobileNetV3Large
)

# =========================
# Evaluation Metrics
# =========================
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, auc,
    classification_report, confusion_matrix
)

# =========================
# External Model Downloading
# =========================
from huggingface_hub import hf_hub_download  # Download models/files from HuggingFace Hub


# =========================
# Version Check (Reproducibility)
# =========================
# Important for ensuring experiment reproducibility.
tf.__version__, keras.__version__



2025-08-16 17:05:31.816731: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-16 17:05:31.825837: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755342331.835758 3651009 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755342331.838658 3651009 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-16 17:05:31.851715: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

('2.18.0', '3.8.0')

In [ ]:
"""
Cell 2: GPU Detection and Device Configuration (You can skip cell 2 and 3 if you are running on CPU-only environment or in online platforms like Google Colab where GPU is automatically managed)
---------------------------------------------

This cell checks for available GPU devices and explicitly selects a specific GPU
for training.

Workflow:
1. Detect all physical GPU devices available to TensorFlow.
2. Print detected GPUs for verification.
3. Restrict TensorFlow to use a specific GPU (e.g., index 1).
4. Handle runtime errors that may occur if devices are already initialized.

This is particularly useful in:
- Multi-GPU systems
- Shared servers
- Controlled experiment environments
"""

# List all available physical GPU devices
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    # Print all detected GPUs
    for i, gpu in enumerate(gpus):
        print(f"GPU {i}: {gpu}")

    try:
        # Select a specific GPU (here: second GPU, index 1)
        # ⚠️ IMPORTANT:
        # Change index [1] if:
        # - You only have one GPU (use gpus[0])
        # - You want to run on a different GPU
        tf.config.set_visible_devices(gpus[1], 'GPU')

        print("Successfully set GPU for training")

    except RuntimeError as e:
        # This error occurs if GPUs have already been initialized
        print("RuntimeError:", e)

"""
Notes:
- The call to `set_visible_devices()` must happen before TensorFlow initializes
  the GPU (i.e., before creating tensors or models).
- If running on a single-GPU system, using gpus[1] will raise an IndexError.
- For dynamic memory allocation (recommended in many cases), you may also use:

    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

  This prevents TensorFlow from allocating all GPU memory at once.
"""

In [ ]:
"""
Cell 3: Verify Visible GPU Devices
----------------------------------

This cell prints the GPU devices currently visible to TensorFlow.

Purpose:
- Confirms whether the GPU selection in the previous cell was applied correctly.
- Ensures TensorFlow is restricted to the intended GPU.
- Helps debug multi-GPU or shared-server configurations.

If `set_visible_devices()` was successfully executed earlier,
this should display only the selected GPU.
"""

import tensorflow as tf

# Retrieve and print currently visible GPU devices
print(tf.config.experimental.get_visible_devices('GPU'))

"""
Expected Behavior:
- If GPU configuration was successful → prints a single GPU (e.g., GPU:1).
- If no GPU is available → prints an empty list [].
- If multiple GPUs appear → restriction may not have been applied correctly.

Note:
`experimental.get_visible_devices()` is safe for verification purposes,
but GPU configuration must always occur BEFORE model/tensor initialization.
"""

[PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [ ]:
"""
Cell 4: Global Configuration and Hyperparameters
------------------------------------------------

This cell defines the core configuration variables used throughout the project.
These act as global hyperparameters and dataset settings for training and evaluation.

Variables:
- num_classes   : Number of output categories for classification.
- batch_size    : Number of samples processed per training step.
- input_size    : Spatial resolution (height & width) of each video frame.
- num_frame     : Number of frames sampled per video clip.
- frame_shape   : Shape of a single input frame (H, W, C).
- BASE_SAVE_PATH: Root directory for dataset storage or experiment outputs.

These values directly affect:
- Model architecture (final Dense layer size)
- GPU memory usage
- Training speed
- Input tensor dimensions
"""

# ⚠️⚠️⚠️⚠️⚠️⚠️⚠️ Please note that the following code except all input dataset already has been preprocessed (i.e. segmentation, cropping etc. should be already done before this code) to have 15 classes, 15 frames per video, and resized to 224x224.

# =========================
# Dataset / Model Parameters
# =========================
num_classes = 15          # ⚠️ Must match the number of dataset categories
batch_size = 4            # Adjust based on GPU memory capacity
input_size = 224          # Standard size for most pretrained CNN backbones
num_frame = 15            # Number of frames sampled per video

# Shape of a single RGB frame
frame_shape = (input_size, input_size, 3)

# =========================
# Data / Checkpoint Path
# =========================
BASE_SAVE_PATH = "/home/CricShot10k"
# ⚠️ Modify this path according to your local dataset directory




In [ ]:
"""
Cell 5: Data Pipeline, Utility Functions, and Training Callbacks
-----------------------------------------------------------------

This cell defines the core data handling pipeline for video classification,
including:

1. DataFrame creation from directory structure
2. Video loading and frame preprocessing
3. tf.data input pipeline construction
4. Sequence visualization utility
5. Training callbacks for optimization control

These functions are intended to remain unchanged to ensure
reproducibility and pipeline consistency across experiments.
"""

# ==========================================================
# DataFrame Creation from Directory Structure
# ==========================================================
def create_dataframe(path, label2id):
    """
    Creates a shuffled Pandas DataFrame containing:
    - Absolute video path
    - Encoded label (integer)
    - Original class name

    Expected directory structure:
        path/
            class_1/
                video1.mp4
                video2.mp4
            class_2/
                video3.mp4
                ...

    Args:
        path (str): Root dataset directory
        label2id (dict): Mapping from class_name → integer label

    Returns:
        pd.DataFrame: Shuffled dataframe with columns:
                      ["video_path", "label", "class_name"]
    """
    data = []

    for class_name in os.listdir(path):
        class_dir = os.path.join(path, class_name)

        if os.path.isdir(class_dir) and class_name in label2id:
            for video_file in os.listdir(class_dir):
                video_path = os.path.join(class_dir, video_file)

                data.append(
                    {
                        "video_path": os.path.abspath(video_path),
                        "label": label2id[class_name],
                        "class_name": class_name,
                    }
                )

    df = pd.DataFrame(data)

    # Shuffle dataset for better training distribution
    df = df.sample(frac=1).reset_index(drop=True)

    return df


# ==========================================================
# Video Reading & Frame Preprocessing
# ==========================================================
def read_video(file_path):
    """
    Reads a full video file using Decord and returns resized frames.

    ⚠️ Note:
    - Loads ALL frames into memory.
    - For long videos, consider sampling frames instead of loading full video.
    """
    vr = VideoReader(file_path.numpy().decode("utf-8"))
    frames = vr.get_batch(range(len(vr))).asnumpy()

    return format_frames(frames, output_size=(input_size, input_size))


def format_frames(frames, output_size):
    """
    Converts frames to uint8 and resizes to target resolution.

    Args:
        frames: Raw video frames
        output_size: (height, width)

    Returns:
        Resized frames tensor
    """
    frames = tf.image.convert_image_dtype(frames, tf.uint8)

    # ⚠️ Resize to match backbone input resolution
    frames = tf.image.resize(frames, size=output_size)

    return frames


def load_video(file_path, label):
    """
    Wraps video loading inside tf.py_function so it can be used
    inside tf.data pipelines.

    Returns:
        video tensor: shape (T, H, W, 3)
        label tensor: float32
    """
    video = tf.py_function(func=read_video, inp=[file_path], Tout=tf.float32)

    # Required because tf.py_function removes static shape info
    video.set_shape([None, None, None, 3])

    return video, tf.cast(label, dtype=tf.float32)


# ==========================================================
# TensorFlow DataLoader
# ==========================================================
def create_dataloader(df, batch_size, shuffle=True, drop_reminder=True):
    """
    Creates an optimized tf.data pipeline.

    Steps:
    - Create dataset from video paths and labels
    - Shuffle (optional)
    - Map video loading
    - Batch
    - Prefetch
    - Repeat (if training)

    Args:
        df: DataFrame from create_dataframe()
        batch_size: Batch size
        shuffle: Whether to shuffle dataset
        drop_reminder: Drop last incomplete batch

    Returns:
        tf.data.Dataset
    """
    ds = tf.data.Dataset.from_tensor_slices(
        (df["video_path"].values, df["label"].values)
    )

    # Shuffle buffer size proportional to batch size
    ds = ds.shuffle(8 * batch_size) if shuffle else ds

    # Parallel video loading
    ds = ds.map(load_video, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(batch_size, drop_remainder=drop_reminder)

    # Performance optimization
    ds = ds.prefetch(tf.data.AUTOTUNE)

    # Repeat for infinite training loop (only when shuffle=True)
    ds = ds.repeat() if shuffle else ds

    return ds


# ==========================================================
# Visualization Utility
# ==========================================================
def show_sequence(seq, sample=8, title=""):
    """
    Displays a sequence of frames horizontally.

    Args:
        seq: Tensor with shape [T, H, W, 3]
        sample: Number of frames to display
        title: Plot title
    """
    assert seq.shape[-1] == 3

    fig = plt.figure(figsize=(20, 2.5))
    fig.suptitle(title, fontsize=16)

    grid = ImageGrid(fig, 111, nrows_ncols=(1, sample), axes_pad=0.1)

    for ax, img in zip(grid, seq):
        ax.imshow(img.astype("uint8"))
        ax.set_axis_off()

    plt.show()


# ==========================================================
# Training Callbacks
# ==========================================================

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    # baseline=None  # Optional: define baseline for stricter control
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.1,     # Reduce LR by 10x
    patience=4,     # Half of early stopping patience
    verbose=1
)


In [ ]:
"""
Cell 6: Model Configuration Grid (Backbone + Temporal Head)
------------------------------------------------------------

This cell defines the experimental search space for training.

It consists of:

1. model_to_train:
   A mapping between integer IDs and CNN backbone names.
   These correspond to pretrained architectures imported earlier
   (used for spatial feature extraction per frame).

2. model_run_list:
   A list of experiment configurations in the format:
       (backbone_id, recurrent_type, recurrent_units)

   Each tuple represents one full experiment:
       - Spatial Backbone (e.g., EfficientNetV2-S)
       - Temporal Modeling Layer (GRU / LSTM / BiLSTM)
       - Number of RNN hidden units

This structure enables systematic ablation studies and
architecture comparisons for the research paper.
"""


# ==========================================================
# Backbone Model Dictionary
# ==========================================================
model_to_train = {
    1: "resnet50",
    2: "resnet50v2",
    3: "densenet121",
    4: "densenet201",
    5: "mobilenetv2",
    6: "mobilenetv3-large",
    7: "efficientnet-b7",
    8: "efficientnetv2-b3",
    9: "efficientnetv2-s",
    10: "inceptionv3",
    11: "inceptionresnetv2",
    12: "xception",
    13: "vgg19",
    14: "nasnet-mobile",
    15: "convnext-tiny"
}



# ==========================================================
# Experiment Run List
# ==========================================================
model_run_list = [

    # Primary backbone experiments (EfficientNetV2-S)
    (9, "GRU", 64),
    (9, "GRU", 128),
    (9, "GRU", 256),
    (9, "LSTM", 128),
    (9, "LSTM", 64),

    # Additional backbone comparisons
    (1, "GRU", 128),   # ResNet50
    (2, "GRU", 128),   # ResNet50V2
    (3, "GRU", 128),   # DenseNet121
    (14, "GRU", 128),  # NASNetMobile
    (15, "GRU", 128),  # ConvNeXtTiny
    (12, "GRU", 128),  # Xception
    (5, "GRU", 128),   # MobileNetV2
    (6, "GRU", 128),   # MobileNetV3-Large
    (11, "GRU", 128),  # InceptionResNetV2
    (8, "GRU", 128),   # EfficientNetV2-B3
    (4, "GRU", 128),   # DenseNet201

    # Bidirectional LSTM Experiments
    (9, "BiLSTM", 128),
    (13, "GRU", 128),  # VGG19
    (10, "GRU", 128),  # InceptionV3
    (7, "GRU", 128),   # EfficientNet-B7

    # BiLSTM scaling study
    (9, "BiLSTM", 32),
    (9, "BiLSTM", 64),
    (9, "BiLSTM", 128),

    # Larger temporal capacity, may go beyond memeory and requires lower batch size
    (9, "LSTM", 256),
]

"""
Interpretation of Tuple:
    (backbone_id, recurrent_layer_type, hidden_units)

Example:
    (9, "GRU", 128)
    → EfficientNetV2-S backbone
    → GRU temporal layer
    → 128 hidden units


"""

In [ ]:
"""
Cell 7: Backbone Model Selector
-------------------------------

This function initializes and loads the pretrained CNN backbone

Parameters
----------
model_num : int
    Integer ID corresponding to a specific CNN architecture.

frame_shape : tuple
    Input tensor shape of a single frame in the format
    (height, width, channels).

Behavior
--------
- Loads ImageNet-pretrained weights
- Removes classification head (include_top=False)
- Returns convolutional feature extractor only
- Raises ValueError if an invalid ID is provided

Returns
-------
base_model : tf.keras.Model
    Pretrained CNN backbone for spatial feature extraction.

"""

def get_base_model(model_num, frame_shape):
    base_model = None

    if model_num == 1:
        base_model = ResNet50(weights='imagenet', include_top=False, input_shape=frame_shape)
    elif model_num == 2:
        base_model = ResNet50V2(weights='imagenet', include_top=False, input_shape=frame_shape)
    elif model_num == 3:
        base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=frame_shape)
    elif model_num == 4:
        base_model = DenseNet201(weights='imagenet', include_top=False, input_shape=frame_shape)
    elif model_num == 5:
        base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=frame_shape)
    elif model_num == 6:
        base_model = MobileNetV3Large(weights='imagenet', include_top=False, input_shape=frame_shape)
    elif model_num == 7:
        base_model = EfficientNetB7(weights='imagenet', include_top=False, input_shape=frame_shape)
    elif model_num == 8:
        base_model = EfficientNetV2B3(weights='imagenet', include_top=False, input_shape=frame_shape)
    elif model_num == 9:
        base_model = EfficientNetV2S(weights='imagenet', include_top=False, input_shape=frame_shape)
    elif model_num == 10:
        base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=frame_shape)
    elif model_num == 11:
        base_model = InceptionResNetV2(weights='imagenet', include_top=False, input_shape=frame_shape)
    elif model_num == 12:
        base_model = Xception(weights='imagenet', include_top=False, input_shape=frame_shape)
    elif model_num == 13:
        base_model = VGG19(weights='imagenet', include_top=False, input_shape=frame_shape)
    elif model_num == 14:
        base_model = NASNetMobile(weights='imagenet', include_top=False, input_shape=frame_shape)
    elif model_num == 15:
        base_model = ConvNeXtTiny(weights='imagenet', include_top=False, input_shape=frame_shape)
    else:
        raise ValueError("Invalid model number. Choose a number between 1 and 15.")

    return base_model

In [ ]:
"""
Cell 8: Spatiotemporal Model Builder (CNN + RNN)
-----------------------------------------------

This function constructs a complete spatiotemporal classification model
by combining a pretrained CNN backbone for spatial feature extraction
with a recurrent layer for temporal sequence modeling.

The configuration is selected from `model_run_list` using `loop_serial`,
where each entry defines:
    (backbone_id, recurrent_type, recurrent_units)

Architecture Pipeline
---------------------
Input Video Clip
    → TimeDistributed(CNN Backbone)
    → TimeDistributed(Flatten)
    → Temporal Layer (GRU / LSTM / BiLSTM)
    → Batch Normalization
    → Fully Connected Layer (ReLU)
    → Softmax Classification Layer

Parameters
----------
loop_serial : int
    Index of the experiment configuration in `model_run_list`.
    Determines backbone architecture and temporal head.

input_shape : tuple
    Shape of the input video clip in the format:
    (num_frames, height, width, channels)

class_count : int
    Number of output classes for classification.

Behavior
--------
- Loads ImageNet-pretrained CNN backbone
- Applies backbone independently to each frame using TimeDistributed
- Aggregates temporal information using RNN (GRU/LSTM/BiLSTM)
- Adds normalization and dense classification layers
- Compiles model using Adam optimizer and cross-entropy loss

Returns
-------
model : tf.keras.Model
    Compiled spatiotemporal classification model ready for training.

"""
def buildModel(loop_serial=0, input_shape=(num_frame, input_size, input_size, 3), class_count=num_classes):
    # Get the config from model_run_list
    model_num, rnn_type, rnn_unit = model_run_list[loop_serial]

    # Get the pretrained CNN base model

    pretrained_base_model = get_base_model(model_num, input_shape[1:])  # shape[1:] gives (size, size, 3)

    if pretrained_base_model is None:
        raise ValueError("Base model could not be loaded.")

    model = Sequential()
    model.add(Input(shape=input_shape))
    model.add(TimeDistributed(pretrained_base_model))
    model.add(TimeDistributed(Flatten()))

    # Choose between GRU and LSTM

    if rnn_type.upper() == "GRU":
        model.add(GRU(rnn_unit))
    elif rnn_type.upper() == "LSTM":
        model.add(LSTM(rnn_unit))
    elif rnn_type.upper() == "BILSTM":
        model.add(Bidirectional(LSTM(rnn_unit)))
    else:
        raise ValueError("Invalid RNN type.")

    model.add(BatchNormalization())
    model.add(Dense(1024, activation='relu'))
    model.add(Dense(class_count, activation='softmax'))

    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=Adam(learning_rate=1e-4),
        metrics=['accuracy']
    )

    return model

In [ ]:
"""
Cell 9: Experiment Training Loop with Resume Capability
------------------------------------------------------

This cell executes the full experimental pipeline across all configurations
defined in `model_run_list`. It supports checkpoint-style resumption, model
training, evaluation, and result persistence for reproducibility.

Workflow
--------
1. Read training progress from `train_state.txt`
2. Resume experiments from last unfinished configuration
3. Build model using selected backbone + temporal head
4. Load dataset and create dataloaders
5. Train model with validation monitoring
6. Save trained model and evaluation metrics
7. Compute Top-1, Top-2, and Top-3 accuracy
8. Generate classification report and confusion matrix
9. Persist results in structured experiment folders
10. Update training state to allow resume on interruption

Key Features
------------
- Resume-safe experiment loop (no retraining completed runs)
- Automatic GPU/CPU memory cleanup between experiments
- Structured result directory per configuration
- Top-k accuracy evaluation (Top-2, Top-3)
- Confusion matrix saved as both text and heatmap
- Classification report export
- Model checkpoint saving

Saved Outputs Per Experiment
----------------------------
- Trained model (.keras)
- classification_report.txt
- confusion_matrix.txt
- confusion_matrix.png
- top_2_3_acuracy.txt


This loop enables systematic large-scale ablation experiments
across multiple backbone and temporal architecture combinations.
"""

# ==========================================================
# Resume Training State
# ==========================================================

train_idx = None

with open(os.path.join(BASE_SAVE_PATH, "train_state.txt"), "r") as f:
    train_idx = int(f.read().strip())


# ==========================================================
# Experiment Loop: Model Build + Data Preparation
# ==========================================================

for i in range(len(model_run_list)):
      if i < train_idx:
          continue

      tf.keras.backend.clear_session()
      gc.collect()

      print(f"\n Loop {i+1} - Config: {model_run_list[i]}")

      input_shape = (num_frame, input_size, input_size, 3)
      model = buildModel(i, input_shape, num_classes)
      model.summary()

      train_set = os.path.join(BASE_SAVE_PATH, "dataset/iutcricshot10k/15fs3it/combined_train")  #change accordingly
      test_set = os.path.join(BASE_SAVE_PATH, "dataset/iutcricshot10k/15fs3it/test")  #change accordingly
      validation_set = os.path.join(BASE_SAVE_PATH, "dataset/iutcricshot10k/15fs3it/validation")  #change accordingly

      excluded_folders = {}

      class_folders = [
          folder for folder in os.listdir(train_set)
          if os.path.isdir(os.path.join(train_set, folder)) and folder not in excluded_folders
      ]

      label2id = {label: i for i, label in enumerate(class_folders)}
      id2label = {v: k for k, v in label2id.items()}

      train_df = create_dataframe(train_set, label2id)
      test_df = create_dataframe(test_set, label2id)
      validation_df = create_dataframe(validation_set, label2id)

      train_ds = create_dataloader(train_df, batch_size, shuffle=True)
      test_ds = create_dataloader(test_df, batch_size, shuffle=False)
      validation_ds = create_dataloader(validation_df, batch_size, shuffle=True)


# ==========================================================
# Model Training
# ==========================================================

      history = model.fit(
          train_ds,
          validation_data=validation_ds,
          steps_per_epoch=len(train_df) // batch_size,
          validation_steps=len(validation_df) // batch_size,
          epochs=100,
          validation_freq=1,
          callbacks=[early_stopping, reduce_lr]
      )


# ==========================================================
# Evaluation, Metrics, and Saving Results
# ==========================================================

      model_num, rnn_type, rnn_unit = model_run_list[i]

      model_name =  model_to_train[model_num]
      folder_name = f"{model_name}_{rnn_type}_{rnn_unit}"    #change accordingly if u want seperate folders to save
      print(f"Saving results in folder: {folder_name}")

      save_folder = os.path.join(BASE_SAVE_PATH, "result", folder_name)

      os.makedirs(save_folder, exist_ok=True)
      model_path = os.path.join(save_folder, f"{os.path.basename(save_folder)}.keras")
      model.save(model_path)

      test_y_list = []
      test_pred_list, test_pred_class_list = [], []
      test_pred_class_top2_list, test_pred_class_top3_list = [], []

      for batch_X, batch_y in test_ds:
          test_y_list.append(batch_y.numpy())

          temp_pred = model.predict(batch_X)
          temp_pred_class = np.argmax(temp_pred, axis=1)
          temp_pred_top2 = np.argsort(temp_pred, axis=1)[:, -2:]
          temp_pred_top3 = np.argsort(temp_pred, axis=1)[:, -3:]

          test_pred_list.append(temp_pred)
          test_pred_class_list.append(temp_pred_class)
          test_pred_class_top2_list.append(temp_pred_top2)
          test_pred_class_top3_list.append(temp_pred_top3)

      test_y = np.concatenate(test_y_list, axis=0)
      test_pred = np.concatenate(test_pred_list, axis=0)
      test_pred_class = np.concatenate(test_pred_class_list, axis=0)
      test_pred_class_top2 = np.concatenate(test_pred_class_top2_list, axis=0)
      test_pred_class_top3 = np.concatenate(test_pred_class_top3_list, axis=0)

      correct_top2 = np.any(test_pred_class_top2 == test_y[:, None], axis=1)
      top2_accuracy = np.mean(correct_top2)

      correct_top3 = np.any(test_pred_class_top3 == test_y[:, None], axis=1)
      top3_accuracy = np.mean(correct_top3)

      with open(os.path.join(save_folder, "top_2_3_acuracy.txt"), "w") as f:
          f.write(f"Top-2 Accuracy:{top2_accuracy}, Top-3 Accuracy: {top3_accuracy}")

      report = classification_report(
          test_y,
          test_pred_class,
          digits=4,
          target_names=[id2label[i] for i in range(len(id2label))]
      )

      with open(os.path.join(save_folder, "classification_report.txt"), "w") as f:
          f.write(report)

      cm = confusion_matrix(test_y, test_pred_class)
      np.savetxt(os.path.join(save_folder, "confusion_matrix.txt"), cm, fmt='%d')

      plt.figure(figsize=(8, 6))
      sns.heatmap(
          cm,
          annot=True,
          cmap="Blues",
          fmt="d",
          xticklabels=[id2label[i] for i in range(len(id2label))],
          yticklabels=[id2label[i] for i in range(len(id2label))]
      )
      plt.xlabel("Predicted Class")
      plt.ylabel("True Class")
      plt.title("Confusion Matrix")
      plt.tight_layout()
      plt.savefig(os.path.join(save_folder, "confusion_matrix.png"))
      plt.close()


# ==========================================================
# Update Resume State + Memory Cleanup
# ==========================================================

      with open(os.path.join(BASE_SAVE_PATH, "train_state.txt"), "w") as f:
        f.write(f"{i+1}")

      del model
      tf.keras.backend.clear_session()
      gc.collect()